# Objetivo

Analizar la evolución del discurso mediático relacionado con:

- Racismo / Xenofobia
- LGTBIfobia

durante los periodos electorales de:

- 2015
- 2016
- 2019 (abril)
- 2019 (noviembre)
- 2023

y comprobar si existe una intensificación de determinados marcos discursivos
en las semanas previas a las elecciones generales.

In [76]:
import pandas as pd
import requests
import time
import re

**Importante:** 

La presencia de estos términos no implica automáticamente discurso de odio. El diccionario se utiliza para identificar temas, marcos discursivos, expresiones hostiles y referencias a violencia o discriminación relacionadas con xenofobia y LGTBIfobia. El análisis mide frecuencia y evolución temporal del lenguaje, no intención del autor

In [77]:
DICCIONARIO = {

    "xenofobia": {

        # Nivel 1: Solo indica que se habla del tema
        "neutro": [
            "inmigración", "migración", "migrante", "migrantes",
            "inmigrante", "inmigrantes", "refugiado", "refugiados",
            "asilo", "frontera", "integración",
            "diversidad cultural", "multiculturalidad",
            "extranjero", "extranjeros"
        ],

        # Nivel 2: Encuadre conflictivo o problematizador
        "marco_conflictivo": [
            "inmigración ilegal",
            "avalancha migratoria",
            "oleada migratoria",
            "presión migratoria",
            "efecto llamada",
            "saturación",
            "colapso de servicios",
            "crisis migratoria",
            "menas",
            "pateras",
            "cayuco",
            "cayucos",
            "repatriación",
            "expulsión masiva",
            "invasión de inmigrantes",
            "flujo migratorio descontrolado",
            "problema migratorio",
            "carga para el sistema",
            "fronteras abiertas",
            "control de fronteras",
            "efecto frontera",
            "inseguridad asociada a la inmigración",
            "delincuencia importada",
            "prioridad nacional",
            "primero los españoles",
            "preferencia nacional",
            "arraigo nacional"
        ],

        # Nivel 3: Hostilidad explícita
        "hostilidad_explicita": [
            "moro",
            "moros",
            "sudaca",
            "sudacas",
            "invasión migratoria",
            "sustitución demográfica",
            "gran sustitución",
            "great replacement",
            "fuera de españa",
            "quédate en tu país",
            "población autóctona en peligro",
            "remigración",
            "islamización",
            "nos invaden",
            "ilegales"
        ],

        # Nivel 4: Violencia o discriminación
        "violencia_discriminacion": [
            "agresión racista",
            "delito de odio racial",
            "discriminación racial",
            "ataque xenófobo",
            "crimen de odio",
            "violencia racista",
            "incidente racista",
            "insulto racista",
            "denuncia por racismo",
            "expulsión discriminatoria"
        ]
    },

    "lgtbifobia": {

        # Nivel 1: Solo indica que se habla del tema
        "neutro": [
            "lgtbi",
            "lgbt",
            "lgtbiq",
            "gay",
            "lesbiana",
            "lesbianas",
            "bisexual",
            "trans",
            "transexual",
            "transexuales",
            "transgénero",
            "identidad de género",
            "orientación sexual",
            "matrimonio homosexual",
            "pareja homosexual",
            "derechos lgtbi"
        ],

        # Nivel 2: Encuadre conflictivo o problematizador
        "marco_conflictivo": [
            "ideología de género",
            "adoctrinamiento",
            "agenda lgtbi",
            "transactivismo",
            "dictadura woke",
            "ingeniería social",
            "sexualización infantil",
            "borrado de las mujeres",
            "familia natural",
            "familia tradicional",
            "terapia de conversión",
            "lobby lgtb",
            "imposición de género",
            "modelo de familia alternativo",
            "adoctrinamiento sexual",
            "propaganda lgtbi",
            "imposición ideológica",
            "activismo trans"
        ],

        # Nivel 3: Hostilidad explícita
        "hostilidad_explicita": [
            "maricón",
            "maricones",
            "bollera",
            "bolleras",
            "travelo",
            "travelos",
            "perversión sexual",
            "anormalidad",
            "enfermedad mental",
            "desviación sexual",
            "depravación",
            "contra natura",
            "aberración"
        ],

        # Nivel 4: Violencia o discriminación
        "violencia_discriminacion": [
            "agresión homófoba",
            "agresión tránsfoba",
            "lgtbifobia",
            "delito de odio",
            "discriminación lgtbi",
            "paliza homófoba",
            "crimen de odio",
            "violencia homófoba",
            "violencia tránsfoba",
            "denuncia por homofobia",
            "transfobia"
        ]
    }
}

## Funciones de análisis NLP

Estas funciones permiten:

- Contar términos del diccionario.
- Clasificar el contenido por categorías.
- Transformar textos en variables numéricas analizables.

Se utilizan expresiones regulares para evitar contar palabras parciales dentro de otras palabras.

In [78]:
def contar_terminos(texto, lista_terminos):

    texto = str(texto).lower()

    total = 0

    for termino in lista_terminos:

        patron = r'\b' + re.escape(termino.lower()) + r'\b'

        total += len(re.findall(patron, texto))

    return total


In [79]:
def analizar_texto(texto, diccionario):

    resultado = {}

    for tema, categorias in diccionario.items():

        for categoria, terminos in categorias.items():

            nombre_columna = f"{tema}_{categoria}"

            resultado[nombre_columna] = contar_terminos(
                texto,
                terminos
            )

    return resultado

In [80]:
def analizar_dataframe(df, columna_texto):

    resultados = df[columna_texto].apply(
        lambda x: analizar_texto(x, DICCIONARIO)
    )

    resultados = pd.DataFrame(resultados.tolist())

    return pd.concat(
        [df.reset_index(drop=True),
         resultados.reset_index(drop=True)],
        axis=1
    )

In [81]:
def consultar_gdelt(query, maxrecords=5):
    time.sleep(6)

    url = (
        "https://api.gdeltproject.org/api/v2/doc/doc"
        f"?query={query}"
        "&mode=ArtList"
        f"&maxrecords={maxrecords}"
        "&format=json"
    )

    respuesta = requests.get(url)

    print(respuesta.status_code)

    if respuesta.status_code == 200:
        return respuesta.json()
    else:
        print(respuesta.text[:500])
        return None

In [82]:
resultado = consultar_gdelt("inmigracion", maxrecords=5)

429
Please limit requests to one every 5 seconds or contact kalev.leetaru5@gmail.com for larger queries. All high-traffic users should switch to our ngrams dataset: https://blog.gdeltproject.org/using-the-new-web-ngrams-dataset-to-find-relevant-coverage/. For trend analysis, please see our daily newsletter briefings: https://blog.gdeltproject.org/announcing-our-new-daily-todays-trends-on-capitol-hill-todays-media-trends-newsletter-briefings/.




In [83]:
def resumen_categoria(df):

    columnas = [
        'xenofobia_neutro',
        'xenofobia_marco_conflictivo',
        'xenofobia_hostilidad_explicita',
        'xenofobia_violencia_discriminacion',
        'lgtbifobia_neutro',
        'lgtbifobia_marco_conflictivo',
        'lgtbifobia_hostilidad_explicita',
        'lgtbifobia_violencia_discriminacion'
    ]

    return df[columnas].sum().sort_values(ascending=False)

## Sistema de ponderación

Se asignan pesos crecientes según el nivel de conflictividad detectado:

- Neutro = 1
- Marco conflictivo = 2
- Violencia/discriminación = 3
- Hostilidad explícita = 4

El objetivo no es medir la gravedad jurídica de una noticia, sino construir un indicador sintético de intensidad discursiva.

In [84]:
PESOS = {
    "neutro": 1,
    "marco_conflictivo": 2,
    "hostilidad_explicita": 4,
    "violencia_discriminacion": 3
}

In [85]:
def calcular_indice_xenofobia(df):

    return (
        df['xenofobia_neutro'] * 1 +
        df['xenofobia_marco_conflictivo'] * 2 +
        df['xenofobia_hostilidad_explicita'] * 4 +
        df['xenofobia_violencia_discriminacion'] * 3
    )

In [86]:
def calcular_indice_lgtbifobia(df):

    return (
        df['lgtbifobia_neutro'] * 1 +
        df['lgtbifobia_marco_conflictivo'] * 2 +
        df['lgtbifobia_hostilidad_explicita'] * 4 +
        df['lgtbifobia_violencia_discriminacion'] * 3
    )

In [87]:
df_resultado['indice_xenofobia'] = calcular_indice_xenofobia(df_resultado)

df_resultado['indice_lgtbifobia'] = calcular_indice_lgtbifobia(df_resultado)

In [88]:
def resumen_indices(df):

    return {
        "frecuencia_xenofobia":
            df["xenofobia_neutro"].sum()
            + df["xenofobia_marco_conflictivo"].sum()
            + df["xenofobia_hostilidad_explicita"].sum()
            + df["xenofobia_violencia_discriminacion"].sum(),

        "frecuencia_lgtbifobia":
            df["lgtbifobia_neutro"].sum()
            + df["lgtbifobia_marco_conflictivo"].sum()
            + df["lgtbifobia_hostilidad_explicita"].sum()
            + df["lgtbifobia_violencia_discriminacion"].sum(),

        "indice_xenofobia":
            df["indice_xenofobia"].sum(),

        "indice_lgtbifobia":
            df["indice_lgtbifobia"].sum()
    }

In [89]:
columnas_noticias = [
    "fecha",
    "medio",
    "titulo",
    "texto",
    "url",
    "tema"
]

df_noticias = pd.DataFrame(columns=columnas_noticias)

df_noticias.head()

,fecha,medio,titulo,texto,url,tema


In [90]:
def pipeline_nlp(df):

    df = analizar_dataframe(df, "texto")

    df["indice_xenofobia"] = calcular_indice_xenofobia(df)

    df["indice_lgtbifobia"] = calcular_indice_lgtbifobia(df)

    return df

In [91]:
def resumen_anual(df):

    return (
        df.groupby("año")
        .agg({
            "indice_xenofobia": "sum",
            "indice_lgtbifobia": "sum",
            "xenofobia_neutro": "sum",
            "xenofobia_marco_conflictivo": "sum",
            "xenofobia_hostilidad_explicita": "sum",
            "lgtbifobia_neutro": "sum",
            "lgtbifobia_marco_conflictivo": "sum",
            "lgtbifobia_hostilidad_explicita": "sum"
        })
    )

Extraer texto de una url

In [92]:
from bs4 import BeautifulSoup
import requests

def extraer_texto_articulo(url):

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    respuesta = requests.get(url, headers=headers, timeout=10)

    if respuesta.status_code != 200:
        return None

    soup = BeautifulSoup(respuesta.text, "html.parser")

    titulo = soup.find("h1")
    titulo = titulo.get_text(strip=True) if titulo else ""

    parrafos = soup.find_all("p")
    texto = " ".join([p.get_text(strip=True) for p in parrafos])

    return {
        "url": url,
        "titulo": titulo,
        "texto": texto
    }

In [93]:
df_articulo = pd.DataFrame([{
    "texto": articulo["texto"]
}])

df_articulo = analizar_dataframe(df_articulo, "texto")

df_articulo["indice_xenofobia"] = calcular_indice_xenofobia(df_articulo)
df_articulo["indice_lgtbifobia"] = calcular_indice_lgtbifobia(df_articulo)

df_articulo[
    [
        "xenofobia_neutro",
        "xenofobia_marco_conflictivo",
        "xenofobia_hostilidad_explicita",
        "xenofobia_violencia_discriminacion",
        "indice_xenofobia",
        "indice_lgtbifobia"
    ]
]

,xenofobia_neutro,xenofobia_marco_conflictivo,xenofobia_hostilidad_explicita,xenofobia_violencia_discriminacion,indice_xenofobia,indice_lgtbifobia
0,3,4,0,0,11,0


In [94]:
def analizar_texto_detallado(texto, diccionario):
    texto = str(texto).lower()

    resultado = {}

    for tema, categorias in diccionario.items():
        for categoria, terminos in categorias.items():

            nombre_conteo = f"{tema}_{categoria}"
            nombre_terminos = f"{tema}_{categoria}_terminos"

            terminos_encontrados = []

            for termino in terminos:
                patron = r'\b' + re.escape(termino.lower()) + r'\b'
                encontrados = re.findall(patron, texto)

                if encontrados:
                    terminos_encontrados.extend(encontrados)

            resultado[nombre_conteo] = len(terminos_encontrados)
            resultado[nombre_terminos] = terminos_encontrados

    return resultado

## Validación del diccionario

Las pruebas manuales muestran que el diccionario detecta correctamente términos neutros, marcos conflictivos, expresiones de hostilidad explícita y referencias a violencia o discriminación, tanto para xenofobia como para LGTBIfobia.

Esto permite aplicar el sistema a noticias reales y convertir texto no estructurado en variables cuantitativas comparables.

In [95]:
url_prueba = "https://www.elmundo.es/andalucia/2026/07/03/6a47bfcde4d4d8e7158b45a9.html"

articulo = extraer_texto_articulo(url_prueba)

articulo

{'url': 'https://www.elmundo.es/andalucia/2026/07/03/6a47bfcde4d4d8e7158b45a9.html',
 'titulo': 'Moreno limita el alcance de la \'prioridad nacional\' pactada con Vox: "Es mentira que se vaya a dejar a migrantes sin sanidad o a los niños se los vaya a desescolarizar"',
 'texto': 'Portada SUSCRÍBETE20%DTO. SUSCRÍBETE20%DTO. SUSCRÍBETE20%DTO. Asegura que pensó en repetir elecciones para que su "vanidad y principios" quedasen "a salvo" pero priorizó el "interés general" Horas después de haber sellado un pacto con Vox que le ha convertido en presidente de la Junta por tercera vez,Juanma Morenointenta explicar el alcance real del acuerdo y hasta dónde llega el concepto más controvertido de su programa de gobierno, el de la"prioridad nacional". Moreno ha aclarado este viernes que el gobierno andaluz no negará ningún servicio público a los inmigrantes: "Es mentira que se vaya a dejar a migrantes sin sanidad o a los niños se los vaya adesescolarizar", ha insistido en una entrevista en la Caden

In [96]:
texto = articulo["texto"]

resultado = analizar_texto(texto, DICCIONARIO)

resultado

{'xenofobia_neutro': 3,
 'xenofobia_marco_conflictivo': 4,
 'xenofobia_hostilidad_explicita': 0,
 'xenofobia_violencia_discriminacion': 0,
 'lgtbifobia_neutro': 0,
 'lgtbifobia_marco_conflictivo': 0,
 'lgtbifobia_hostilidad_explicita': 0,
 'lgtbifobia_violencia_discriminacion': 0}

In [97]:
analisis_detallado = analizar_texto_detallado(
    articulo["texto"],
    DICCIONARIO
)

analisis_detallado

{'xenofobia_neutro': 3,
 'xenofobia_neutro_terminos': ['migrantes', 'inmigrantes', 'inmigrantes'],
 'xenofobia_marco_conflictivo': 4,
 'xenofobia_marco_conflictivo_terminos': ['prioridad nacional',
  'prioridad nacional',
  'prioridad nacional',
  'primero los españoles'],
 'xenofobia_hostilidad_explicita': 0,
 'xenofobia_hostilidad_explicita_terminos': [],
 'xenofobia_violencia_discriminacion': 0,
 'xenofobia_violencia_discriminacion_terminos': [],
 'lgtbifobia_neutro': 0,
 'lgtbifobia_neutro_terminos': [],
 'lgtbifobia_marco_conflictivo': 0,
 'lgtbifobia_marco_conflictivo_terminos': [],
 'lgtbifobia_hostilidad_explicita': 0,
 'lgtbifobia_hostilidad_explicita_terminos': [],
 'lgtbifobia_violencia_discriminacion': 0,
 'lgtbifobia_violencia_discriminacion_terminos': []}

In [98]:
texto_hostil = """
La invasión migratoria continúa.
Muchos afirman que nos invaden.
Algunos piden la remigración y dicen
que los ilegales deberían volver a su país.
"""

analizar_texto_detallado(
    texto_hostil,
    DICCIONARIO
)

{'xenofobia_neutro': 0,
 'xenofobia_neutro_terminos': [],
 'xenofobia_marco_conflictivo': 0,
 'xenofobia_marco_conflictivo_terminos': [],
 'xenofobia_hostilidad_explicita': 4,
 'xenofobia_hostilidad_explicita_terminos': ['invasión migratoria',
  'remigración',
  'nos invaden',
  'ilegales'],
 'xenofobia_violencia_discriminacion': 0,
 'xenofobia_violencia_discriminacion_terminos': [],
 'lgtbifobia_neutro': 0,
 'lgtbifobia_neutro_terminos': [],
 'lgtbifobia_marco_conflictivo': 0,
 'lgtbifobia_marco_conflictivo_terminos': [],
 'lgtbifobia_hostilidad_explicita': 0,
 'lgtbifobia_hostilidad_explicita_terminos': [],
 'lgtbifobia_violencia_discriminacion': 0,
 'lgtbifobia_violencia_discriminacion_terminos': []}

In [99]:
texto_lgtbi = """
La ideología de género está destruyendo la familia tradicional.
Algunos consideran que el transactivismo es una imposición ideológica.

Otros utilizan términos como maricón o travelo.

Además se ha producido una agresión homófoba.
"""

analizar_texto_detallado(
    texto_lgtbi,
    DICCIONARIO
)

{'xenofobia_neutro': 0,
 'xenofobia_neutro_terminos': [],
 'xenofobia_marco_conflictivo': 0,
 'xenofobia_marco_conflictivo_terminos': [],
 'xenofobia_hostilidad_explicita': 0,
 'xenofobia_hostilidad_explicita_terminos': [],
 'xenofobia_violencia_discriminacion': 0,
 'xenofobia_violencia_discriminacion_terminos': [],
 'lgtbifobia_neutro': 0,
 'lgtbifobia_neutro_terminos': [],
 'lgtbifobia_marco_conflictivo': 4,
 'lgtbifobia_marco_conflictivo_terminos': ['ideología de género',
  'transactivismo',
  'familia tradicional',
  'imposición ideológica'],
 'lgtbifobia_hostilidad_explicita': 2,
 'lgtbifobia_hostilidad_explicita_terminos': ['maricón', 'travelo'],
 'lgtbifobia_violencia_discriminacion': 1,
 'lgtbifobia_violencia_discriminacion_terminos': ['agresión homófoba']}

In [100]:
df_noticias = pd.DataFrame([
    {
        "medio": "El Mundo",
        "fecha": "2026-07-03",
        "tema": "xenofobia",
        "url": "URL_1",
        "titulo": "Título noticia",
        "texto": "Texto completo noticia"
    }
])

In [101]:
noticias = [
    {
        "medio": "El Mundo",
        "fecha": "2026-07-03",
        "tema": "xenofobia",
        "url": articulo["url"],
        "titulo": articulo["titulo"],
        "texto": articulo["texto"]
    }
]

df_noticias = pd.DataFrame(noticias)

df_noticias.head()

,medio,fecha,tema,url,titulo,texto
0,El Mundo,2026-07-03,xenofobia,https://www.elmundo.es/andalucia/2026/07/03/6a...,Moreno limita el alcance de la 'prioridad naci...,Portada SUSCRÍBETE20%DTO. SUSCRÍBETE20%DTO. SU...


In [102]:
df_noticias = pipeline_nlp(df_noticias)

df_noticias[
    [
        "medio",
        "titulo",
        "indice_xenofobia",
        "indice_lgtbifobia"
    ]
]

,medio,titulo,indice_xenofobia,indice_lgtbifobia
0,El Mundo,Moreno limita el alcance de la 'prioridad naci...,11,0


In [110]:
url_abc = "https://www.abc.es/espana/andalucia/rechazo-extranjeros-sube-delitos-odio-andalucia-20260603175633-nts.html"

articulo_abc = extraer_texto_articulo(url_abc)

articulo_abc


url_elpais = "https://elpais.com/internacional/2026-07-01/la-policia-britanica-investiga-por-negligencia-grave-a-dos-de-los-agentes-que-arrestaron-a-henry-nowak.html"

articulo_elpais = extraer_texto_articulo(url_elpais)

articulo_elpais

url_elmundo = "https://www.elmundo.es/andalucia/2026/07/03/6a47bfcde4d4d8e7158b45a9.html"

articulo_elmundo = extraer_texto_articulo(url_elmundo)

articulo_elmundo

{'url': 'https://www.elmundo.es/andalucia/2026/07/03/6a47bfcde4d4d8e7158b45a9.html',
 'titulo': 'Moreno limita el alcance de la \'prioridad nacional\' pactada con Vox: "Es mentira que se vaya a dejar a migrantes sin sanidad o a los niños se los vaya a desescolarizar"',
 'texto': 'Portada SUSCRÍBETE20%DTO. SUSCRÍBETE20%DTO. SUSCRÍBETE20%DTO. Asegura que pensó en repetir elecciones para que su "vanidad y principios" quedasen "a salvo" pero priorizó el "interés general" Horas después de haber sellado un pacto con Vox que le ha convertido en presidente de la Junta por tercera vez,Juanma Morenointenta explicar el alcance real del acuerdo y hasta dónde llega el concepto más controvertido de su programa de gobierno, el de la"prioridad nacional". Moreno ha aclarado este viernes que el gobierno andaluz no negará ningún servicio público a los inmigrantes: "Es mentira que se vaya a dejar a migrantes sin sanidad o a los niños se los vaya adesescolarizar", ha insistido en una entrevista en la Caden

In [111]:
noticias = []

if articulo_abc is not None:
    noticias.append({
        "medio": "ABC",
        "fecha": "2026-06-03",
        "tema": "xenofobia",
        "url": articulo_abc["url"],
        "titulo": articulo_abc["titulo"],
        "texto": articulo_abc["texto"]
    })

if articulo_elpais is not None:
    noticias.append({
        "medio": "El País",
        "fecha": "2026-07-01",
        "tema": "xenofobia",
        "url": articulo_elpais["url"],
        "titulo": articulo_elpais["titulo"],
        "texto": articulo_elpais["texto"]
    })

if articulo_elmundo is not None:
    noticias.append({
        "medio": "El Mundo",
        "fecha": "2026-07-03",
        "tema": "xenofobia",
        "url": articulo_elmundo["url"],
        "titulo": articulo_elmundo["titulo"],
        "texto": articulo_elmundo["texto"]
    })

df_noticias = pd.DataFrame(noticias)

df_noticias[["medio", "fecha", "titulo"]]

,medio,fecha,titulo
0,ABC,2026-06-03,El rechazo a los extranjeros sube los delitos ...
1,El Mundo,2026-07-03,Moreno limita el alcance de la 'prioridad naci...


In [112]:
print(type(articulo_elmundo))
print(type(articulo_abc))
print(type(articulo_elpais))

<class 'dict'>
<class 'dict'>
<class 'NoneType'>


In [113]:
df_noticias = pipeline_nlp(df_noticias)

df_noticias[
    [
        "medio",
        "xenofobia_neutro",
        "xenofobia_marco_conflictivo",
        "xenofobia_hostilidad_explicita",
        "xenofobia_violencia_discriminacion",
        "indice_xenofobia"
    ]
]

,medio,xenofobia_neutro,xenofobia_marco_conflictivo,xenofobia_hostilidad_explicita,xenofobia_violencia_discriminacion,indice_xenofobia
0,ABC,5,0,0,0,5
1,El Mundo,3,4,0,0,11


In [114]:
df_noticias[
    [
        "medio",
        "titulo",
        "indice_xenofobia"
    ]
].sort_values(
    "indice_xenofobia",
    ascending=False
)

,medio,titulo,indice_xenofobia
1,El Mundo,Moreno limita el alcance de la 'prioridad naci...,11
0,ABC,El rechazo a los extranjeros sube los delitos ...,5


In [115]:
for i, fila in df_noticias.iterrows():
    print("\n" + "="*80)
    print(fila["medio"])
    print(fila["titulo"])
    print(analizar_texto_detallado(fila["texto"], DICCIONARIO))


ABC
El rechazo a los extranjeros sube los delitos de odio en Andalucía
{'xenofobia_neutro': 5, 'xenofobia_neutro_terminos': ['extranjeros', 'extranjeros', 'extranjeros', 'extranjeros', 'extranjeros'], 'xenofobia_marco_conflictivo': 0, 'xenofobia_marco_conflictivo_terminos': [], 'xenofobia_hostilidad_explicita': 0, 'xenofobia_hostilidad_explicita_terminos': [], 'xenofobia_violencia_discriminacion': 0, 'xenofobia_violencia_discriminacion_terminos': [], 'lgtbifobia_neutro': 4, 'lgtbifobia_neutro_terminos': ['orientación sexual', 'orientación sexual', 'orientación sexual', 'orientación sexual'], 'lgtbifobia_marco_conflictivo': 0, 'lgtbifobia_marco_conflictivo_terminos': [], 'lgtbifobia_hostilidad_explicita': 0, 'lgtbifobia_hostilidad_explicita_terminos': [], 'lgtbifobia_violencia_discriminacion': 0, 'lgtbifobia_violencia_discriminacion_terminos': []}

El Mundo
Moreno limita el alcance de la 'prioridad nacional' pactada con Vox: "Es mentira que se vaya a dejar a migrantes sin sanidad o a l

In [116]:
df_noticias["frecuencia_xenofobia"] = (
    df_noticias["xenofobia_neutro"]
    + df_noticias["xenofobia_marco_conflictivo"]
    + df_noticias["xenofobia_hostilidad_explicita"]
    + df_noticias["xenofobia_violencia_discriminacion"]
)

df_noticias["frecuencia_lgtbifobia"] = (
    df_noticias["lgtbifobia_neutro"]
    + df_noticias["lgtbifobia_marco_conflictivo"]
    + df_noticias["lgtbifobia_hostilidad_explicita"]
    + df_noticias["lgtbifobia_violencia_discriminacion"]
)

In [117]:
df_noticias[
    [
        "medio",
        "frecuencia_xenofobia",
        "indice_xenofobia",
        "frecuencia_lgtbifobia",
        "indice_lgtbifobia"
    ]
]

,medio,frecuencia_xenofobia,indice_xenofobia,frecuencia_lgtbifobia,indice_lgtbifobia
0,ABC,5,5,4,4
1,El Mundo,7,11,0,0


In [118]:
urls_noticias = [
    {
        "medio": "La Vanguardia",
        "fecha": "2026-07-02",
        "tema": "xenofobia",
        "url": "https://www.lavanguardia.com/politica/20260702/11582989/acuerdo-pp-vox-andalucia-incluye-150-medidas-prioridad-nacional-auditoria-gasto-inmigracion.html"
    },
    {
        "medio": "La Vanguardia",
        "fecha": "2026-07-02",
        "tema": "xenofobia",
        "url": "https://www.lavanguardia.com/politica/20260702/11582999/documento-acuerdo-gobierno-andalucia-pp-vox.html"
    },
    {
        "medio": "elDiario.es",
        "fecha": "2026-06-18",
        "tema": "lgtbifobia",
        "url": "https://www.eldiario.es/murcia/politica/diputado-murciano-vox-comunismo-sacado-conductas-lgtbi-catalogo-trastornos-metido-derechos-humanos_1_13312557.html"
    },
    {
        "medio": "elDiario.es",
        "fecha": "2026-06-17",
        "tema": "lgtbifobia",
        "url": "https://www.eldiario.es/sociedad/ley-castigar-carcel-falsas-terapias-conversion-lgtbi-avanza-congreso-oposicion-pp-vox_1_13313044.html"
    }
]

In [119]:
noticias_extraidas = []

for noticia in urls_noticias:
    articulo = extraer_texto_articulo(noticia["url"])

    if articulo is not None:
        noticias_extraidas.append({
            "medio": noticia["medio"],
            "fecha": noticia["fecha"],
            "tema": noticia["tema"],
            "url": articulo["url"],
            "titulo": articulo["titulo"],
            "texto": articulo["texto"]
        })

df_nuevas = pd.DataFrame(noticias_extraidas)

df_nuevas[["medio", "fecha", "tema", "titulo"]]

,medio,fecha,tema,titulo
0,La Vanguardia,2026-07-02,xenofobia,El acuerdo entre PP y Vox en Andalucía incluye...
1,La Vanguardia,2026-07-02,xenofobia,Documento: el acuerdo para el Gobierno de Anda...
2,elDiario.es,2026-06-18,lgtbifobia,Un diputado murciano de Vox: “El comunismo ha ...
3,elDiario.es,2026-06-17,lgtbifobia,La ley para castigar con cárcel las falsas ter...


In [120]:
df_nuevas = pipeline_nlp(df_nuevas)

df_nuevas[
    [
        "medio",
        "tema",
        "indice_xenofobia",
        "indice_lgtbifobia"
    ]
]

,medio,tema,indice_xenofobia,indice_lgtbifobia
0,La Vanguardia,xenofobia,11,0
1,La Vanguardia,xenofobia,16,0
2,elDiario.es,lgtbifobia,6,15
3,elDiario.es,lgtbifobia,4,19


In [121]:
df_total_noticias = pd.concat(
    [df_noticias, df_nuevas],
    ignore_index=True
)

df_total_noticias[
    [
        "medio",
        "tema",
        "indice_xenofobia",
        "indice_lgtbifobia"
    ]
]

,medio,tema,indice_xenofobia,indice_lgtbifobia
0,ABC,xenofobia,5,4
1,El Mundo,xenofobia,11,0
2,La Vanguardia,xenofobia,11,0
3,La Vanguardia,xenofobia,16,0
4,elDiario.es,lgtbifobia,6,15
5,elDiario.es,lgtbifobia,4,19


In [122]:
df_total_noticias.groupby("tema")[
    [
        "indice_xenofobia",
        "indice_lgtbifobia"
    ]
].mean()

,indice_xenofobia,indice_lgtbifobia
tema,,
lgtbifobia,5.00,17.0
xenofobia,10.75,1.0


In [123]:
import requests
import time
import pandas as pd
from urllib.parse import quote

def consultar_gdelt_seguro(query, maxrecords=50, intentos=3):
    query_encoded = quote(query)

    url = (
        "https://api.gdeltproject.org/api/v2/doc/doc"
        f"?query={query_encoded}"
        "&mode=ArtList"
        f"&maxrecords={maxrecords}"
        "&format=json"
    )

    for intento in range(intentos):
        time.sleep(7)

        respuesta = requests.get(url)

        if respuesta.status_code == 200:
            return respuesta.json()

        if respuesta.status_code == 429:
            print("Rate limit. Esperando 60 segundos...")
            time.sleep(60)
        else:
            print("Error:", respuesta.status_code)
            print(respuesta.text[:300])
            return None

    return None

In [124]:
resultado = consultar_gdelt_seguro("inmigracion sourceCountry:SP", maxrecords=20)

In [125]:
resultado

{}

In [126]:
print(type(resultado))
print(resultado)

<class 'dict'>
{}


In [127]:
resultado = consultar_gdelt_seguro(
    "inmigracion",
    maxrecords=10
)

resultado

Rate limit. Esperando 60 segundos...
Rate limit. Esperando 60 segundos...
Rate limit. Esperando 60 segundos...
